In [91]:
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

from xgboost import XGBClassifier
import joblib


In [92]:
# 1. Reproducibility
# =========================
random.seed(42)
np.random.seed(42)

In [93]:
# 2. Load dataset
# =========================
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

In [94]:
# 3. CLEAN DATA (IMPORTANT IMPROVEMENT)
# =========================

# Remove missing / invalid values
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df = df.dropna()

# Convert types
df['TotalCharges'] = df['TotalCharges'].astype(float)

# Drop ID (noise)
df = df.drop(columns=['customerID'])

# Target encoding
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [95]:
# 4. FEATURE ENGINEERING (BIG IMPROVEMENT)
# =========================

df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)
df['IsNewCustomer'] = (df['tenure'] < 12).astype(int)
df['HighChargeFlag'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

In [96]:
# 5. SPLIT DATA (NO LEAKAGE)
# =========================
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [97]:
# 6. SMART PREPROCESSING
# =========================
numeric_features = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'AvgMonthlySpend'
]

categorical_features = [
    col for col in X.columns if col not in numeric_features
]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
])


X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [98]:
# 7. MODEL (STRONG & BALANCED)
# =========================
ratio = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

model = XGBClassifier(
    n_estimators=700,
    learning_rate=0.02,
    max_depth=4,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1,
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.02, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=700, n_jobs=None,
              num_parallel_tree=None, ...)

In [99]:
# 8. PREDICTION
# =========================
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

In [100]:
# 9. EVALUATION
# =========================
print("\n===== FINAL CLEAN MODEL RESULTS =====")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


===== FINAL CLEAN MODEL RESULTS =====
Accuracy : 0.738450604122246
Precision: 0.5051724137931034
Recall   : 0.7834224598930482
F1-score : 0.6142557651991615
ROC-AUC  : 0.8346167385373582

Confusion Matrix:
[[746 287]
 [ 81 293]]


In [101]:
# 10. SAVE MODEL
# =========================
joblib.dump(model, "final_xgb_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

print("\nModel saved successfully.")


Model saved successfully.
